# Automatic Ticket Classification - NLP Case Study

**Author:** Praveen Chand  
**Date:** October 2026

## Problem Statement
Build a model to automatically classify customer complaints into 5 categories using NMF and supervised learning.

In [1]:
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Data

In [2]:
# Load JSON data
with open('complaints-2021-05-14_08_16.json', 'r') as f:
    data = json.load(f)

# Extract _source from each record
records = [item['_source'] for item in data]
df = pd.DataFrame(records)

print(f"✓ Loaded {len(df)} records")
print(f"Columns: {len(df.columns)}")
df.head()

✓ Loaded 78313 records
Columns: 18


,tags,zip_code,complaint_id,issue,date_received,state,consumer_disputed,product,company_response,company,submitted_via,date_sent_to_company,company_public_response,sub_product,timely,complaint_what_happened,sub_issue,consumer_consent_provided
0,NaN,90301,3211475,Attempts to collect debt not owed,2019-04-13T12:00:00-05:00,CA,N/A,Debt collection,Closed with explanation,JPMORGAN CHASE & CO.,Web,2019-04-13T12:00:00-05:00,NaN,Credit card debt,Yes,,Debt is not yours,Consent not provided
1,Servicemember,319XX,3229299,Written notification about debt,2019-05-01T12:00:00-05:00,GA,N/A,Debt collection,Closed with explanation,JPMORGAN CHASE & CO.,Web,2019-05-01T12:00:00-05:00,NaN,Credit card debt,Yes,Good morning my name is XXXX XXXX and I apprec...,Didn't receive enough information to verify debt,Consent provided
2,NaN,77069,3199379,"Other features, terms, or problems",2019-04-02T12:00:00-05:00,TX,N/A,Credit card or prepaid card,Closed with explanation,JPMORGAN CHASE & CO.,Web,2019-04-02T12:00:00-05:00,NaN,General-purpose credit card or charge card,Yes,I upgraded my XXXX XXXX card in XX/XX/2018 and...,Problem with rewards from credit card,Consent provided
3,NaN,48066,2673060,Trouble during payment process,2017-09-13T12:00:00-05:00,MI,N/A,Mortgage,Closed with explanation,JPMORGAN CHASE & CO.,Web,2017-09-14T12:00:00-05:00,NaN,Conventional home mortgage,Yes,,NaN,Consent not provided
4,NaN,10473,3203545,Fees or interest,2019-04-05T12:00:00-05:00,NY,N/A,Credit card or prepaid card,Closed with explanation,JPMORGAN CHASE & CO.,Referral,2019-04-05T12:00:00-05:00,NaN,General-purpose credit card or charge card,Yes,,Charged too much interest,N/A


## 2. Data Cleaning

In [3]:
# Use complaint_what_happened as text
df_clean = df.copy()
df_clean['complaint_text'] = df_clean['complaint_what_happened'].fillna('')

# Remove empty and duplicates
df_clean = df_clean[df_clean['complaint_text'].str.len() > 10]
df_clean = df_clean.drop_duplicates(subset=['complaint_text'])
df_clean.reset_index(drop=True, inplace=True)

print(f"Cleaned: {len(df_clean)} records")
print(f"\nProduct distribution:")
df_clean['product'].value_counts().head(10)

Cleaned: 20930 records

Product distribution:


product
Credit card or prepaid card                                                     5024
Checking or savings account                                                     4286
Mortgage                                                                        3246
Credit card                                                                     2068
Credit reporting, credit repair services, or other personal consumer reports    1997
Bank account or service                                                         1649
Debt collection                                                                  930
Money transfer, virtual currency, or money service                               797
Vehicle loan or lease                                                            399
Consumer Loan                                                                    239
Name: count, dtype: int64

## 3. Text Preprocessing

In [4]:
import re, string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data
for res in ['stopwords', 'punkt', 'wordnet', 'punkt_tab', 'averaged_perceptron_tagger', 'omw-1.4']:
    try:
        nltk.data.find(f'corpora/{res}')
    except:
        nltk.download(res, quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

print("✓ NLTK resources ready")

✓ NLTK resources ready


In [5]:
def preprocess_text(text):
    if not isinstance(text, str) or len(text) < 10:
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', '', text)
    text = re.sub(r'\$\d+(\.\d{2})?', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

# Test on sample
sample = df_clean['complaint_text'].iloc[5]
print(f"Original: {sample[:200]}...")
print(f"\nProcessed: {preprocess_text(sample)[:200]}...")

Original: Can you please remove inquiry...



Processed: please remove inquiry...


In [6]:
print("Processing all texts...")
df_clean['processed_text'] = df_clean['complaint_text'].apply(preprocess_text)
df_clean = df_clean[df_clean['processed_text'].str.len() > 0]
df_clean.reset_index(drop=True, inplace=True)
print(f"✓ {len(df_clean)} records processed")

Processing all texts...


✓ 20930 records processed


## 4. Feature Extraction (TF-IDF)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000, min_df=5, max_df=0.8, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_clean['processed_text'])

print(f"TF-IDF Matrix: {tfidf_matrix.shape}")
print(f"Sparsity: {(1.0 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.2%}")

TF-IDF Matrix: (20930, 3000)
Sparsity: 97.39%


## 5. Topic Modeling (NMF)

In [8]:
from sklearn.decomposition import NMF

nmf_model = NMF(n_components=5, random_state=42, max_iter=500)
nmf_features = nmf_model.fit_transform(tfidf_matrix)

print("✓ NMF model trained")
print(f"Reconstruction error: {nmf_model.reconstruction_err_:.2f}")

# Show topics
feature_names = tfidf.get_feature_names_out()
for i, topic in enumerate(nmf_model.components_):
    top_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"\nTopic {i}: {' | '.join(top_words)}")

✓ NMF model trained
Reconstruction error: 135.88

Topic 0: xxxxxxxx | charge | dispute | card | transaction | claim | called | call | told | would

Topic 1: xxxx xxxx | bank | chase bank | xxxx chase | xxxxxxxx xxxx | xxxx xxxxxxxx | name | number | morgan | chase xxxx

Topic 2: credit | card | credit card | report | credit report | inquiry | account | score | chase credit | balance

Topic 3: payment | loan | mortgage | late | interest | modification | home | xxxxxxxx | month | year

Topic 4: account | bank | check | money | fund | chase bank | checking | deposit | closed | checking account


In [9]:
# Assign topics
dominant_topics = np.argmax(nmf_features, axis=1)
df_clean['topic'] = dominant_topics

# Map to categories
topic_labels = {
    0: 'Credit card / Prepaid card',
    1: 'Bank account services',
    2: 'Theft/Dispute reporting',
    3: 'Mortgages/loans',
    4: 'Others'
}

df_clean['category'] = df_clean['topic'].map(topic_labels)

print("Category distribution:")
df_clean['category'].value_counts()

Category distribution:


category
Others                        5204
Theft/Dispute reporting       4735
Mortgages/loans               4524
Credit card / Prepaid card    3291
Bank account services         3176
Name: count, dtype: int64

## 6. Supervised Learning

In [10]:
from sklearn.model_selection import train_test_split

X = df_clean['processed_text']
y = df_clean['topic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# New TF-IDF for supervised
tfidf_sup = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
X_train_tfidf = tfidf_sup.fit_transform(X_train)
X_test_tfidf = tfidf_sup.transform(X_test)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 16744, Test: 4186


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=15, random_state=42),
    'Naive Bayes': MultinomialNB()
}

results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_tfidf, y_train)
    test_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred, average='weighted')
    rec = recall_score(y_test, test_pred, average='weighted')
    f1 = f1_score(y_test, test_pred, average='weighted')
    
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    print(f"  Accuracy: {acc:.4f}")

pd.DataFrame(results).T


Training Logistic Regression...


  Accuracy: 0.9627

Training Random Forest...


  Accuracy: 0.8765

Training Decision Tree...


  Accuracy: 0.7900

Training Naive Bayes...
  Accuracy: 0.7991


,Accuracy,Precision,Recall,F1
Logistic Regression,0.962733,0.962748,0.962733,0.962706
Random Forest,0.876493,0.879099,0.876493,0.875180
Decision Tree,0.790014,0.792133,0.790014,0.790679
Naive Bayes,0.799092,0.802381,0.799092,0.790469


## 7. Model Comparison & Best Model

In [12]:
results_df = pd.DataFrame(results).T
best = results_df['Accuracy'].idxmax()
print(f"🏆 Best Model: {best}")
print(f"Accuracy: {results_df.loc[best, 'Accuracy']:.4f}")
results_df

🏆 Best Model: Logistic Regression
Accuracy: 0.9627


,Accuracy,Precision,Recall,F1
Logistic Regression,0.962733,0.962748,0.962733,0.962706
Random Forest,0.876493,0.879099,0.876493,0.875180
Decision Tree,0.790014,0.792133,0.790014,0.790679
Naive Bayes,0.799092,0.802381,0.799092,0.790469


## 8. Model Inference

In [13]:
def predict(text):
    proc = preprocess_text(text)
    vec = tfidf_sup.transform([proc])
    pred = models['Logistic Regression'].predict(vec)[0]
    prob = models['Logistic Regression'].predict_proba(vec)[0]
    return topic_labels[pred], prob[pred]

tests = [
    "I lost my credit card and someone made unauthorized charges",
    "My mortgage loan was rejected",
    "Unauthorized withdrawal from checking account"
]

for t in tests:
    cat, conf = predict(t)
    print(f"\nText: {t[:60]}...")
    print(f"→ {cat} ({conf:.2%} confidence)")


Text: I lost my credit card and someone made unauthorized charges...
→ Theft/Dispute reporting (96.02% confidence)

Text: My mortgage loan was rejected...
→ Mortgages/loans (99.92% confidence)

Text: Unauthorized withdrawal from checking account...
→ Others (99.70% confidence)


## 9. Save Models

In [14]:
import pickle

with open('best_model.pkl', 'wb') as f:
    pickle.dump(models['Logistic Regression'], f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_sup, f)
with open('nmf_model.pkl', 'wb') as f:
    pickle.dump(nmf_model, f)

print("✓ Models saved successfully!")

✓ Models saved successfully!
